# Ideal reference: VQC OpenQASM (`things_to_run_on_ibm`)

Noiseless **statevector** simulation of each `qvc_test_*.qasm` file. Use the CSV to join against hardware runs (same normalised QASM / same test index).

**Install (once):** `pip install qiskit qiskit_qasm3_import`

**Outputs:** `vqc_ideal_reference_from_qasm.csv` in this folder (`analysis/`).

**Columns (for analysis):**
- `true_label` — parsed from the QASM header comment (`true label ±1`).
- `ideal_Z_all_expectation` — exact $\langle Z^{\otimes 10}\rangle$ (same quantity as hardware parity readout).
- `ideal_prediction` — `sign` of that expectation (`+1` / `-1`), comparable to sampler-based class.
- `ideal_entropy_bits`, `ideal_linear_xeb_purity` ($2^n\sum p^2$), `ideal_top_bitstring`, `ideal_mean_hamming` — characterise the **ideal** output distribution in the computational basis (before readout).


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from qiskit import qasm3
from qiskit.quantum_info import SparsePauliOp, Statevector

def _qasm_dir() -> Path:
    here = Path.cwd().resolve()
    for base in (here, here.parent):
        cand = base / "qnn_code" / "things_to_run_on_ibm"
        if cand.is_dir():
            return cand
    raise FileNotFoundError(
        "Could not find experiments/things_to_run_on_ibm — run from repo root or analysis/"
    )


QASM_DIR = _qasm_dir()


def _out_dir() -> Path:
    here = Path.cwd().resolve()
    if here.name == "code_data_analysis":
        return here
    sub = here / "code_data_analysis"
    if sub.is_dir():
        return sub
    return here


OUT_CSV = _out_dir() / "vqc_ideal_reference_from_qasm.csv"


def _parse_true_label(qasm_text: str) -> float | None:
    for line in qasm_text.splitlines()[:5]:
        m = re.search(r"true\s+label\s*(-?\d+)", line, re.I)
        if m:
            return float(m.group(1))
    return None


def _distribution_stats(probs: dict[str, float], n: int) -> dict:
    p = np.array(list(probs.values()), dtype=float)
    p = p[p > 0]
    entropy = float(-(p * np.log2(p)).sum())
    hog = max(probs.items(), key=lambda kv: kv[1])
    mean_hw = 0.0
    for bitstr, pr in probs.items():
        mean_hw += pr * bitstr.count("1")
    return {
        "ideal_entropy_bits": entropy,
        "ideal_linear_xeb_purity": float((2**n) * float(np.sum(np.array(list(probs.values())) ** 2))),
        "ideal_top_bitstring": str(hog[0]),
        "ideal_top_prob": float(hog[1]),
        "ideal_mean_hamming": float(mean_hw),
    }


def analyse_vqc_qasm(path: Path) -> dict:
    text = path.read_text()
    qc = qasm3.loads(text)
    qc_u = qc.remove_final_measurements(inplace=False)
    n = qc_u.num_qubits
    sv = Statevector(qc_u)
    obs = SparsePauliOp("Z" * n)
    ev = complex(sv.expectation_value(obs)).real
    probs = {str(k): float(v) for k, v in sv.probabilities_dict().items()}
    stats = _distribution_stats(probs, n)
    m = re.search(r"qvc_test_(\d+)", path.name)
    test_index = int(m.group(1)) if m else -1
    tl = _parse_true_label(text)
    pred = 1 if ev >= 0 else -1
    row = {
        "qasm_file": path.name,
        "qasm_path": str(path),
        "test_index": test_index,
        "true_label": tl,
        "n_qubits": n,
        "ideal_Z_all_expectation": ev,
        "ideal_prediction": pred,
        "ideal_prediction_matches_label": (pred == int(tl)) if tl is not None else np.nan,
    }
    row.update(stats)
    return row


paths = sorted(QASM_DIR.glob("qvc_test_*.qasm"))
assert paths, f"No qvc_test_*.qasm under {QASM_DIR}"
rows = [analyse_vqc_qasm(p) for p in paths]
df = pd.DataFrame(rows).sort_values("test_index").reset_index(drop=True)
df.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV.resolve()}  ({len(df)} rows)")
df


## Using this with hardware CSVs

**VQC:** Join `vqc_ideal_reference_from_qasm.csv` to per-device results on `test_index` (same as `qvc_test_NNN`) or on `qasm_file`. Compare `ideal_Z_all_expectation` to the sampler estimate of $\langle Z^{10}\rangle$, and `ideal_prediction` / `true_label` to hardware `prediction` once you align `job_id` ↔ test image.

**XEB:** Join on `qasm_file` or (`depth_key`, `instance_key`). Compare empirical bitstring entropy / purity / TV distance to hardware using `ideal_*` columns as the noiseless reference distribution for that circuit.